In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Figure 6D–H: three-gene internal classification benchmark and ablation.

Input: Seurat RNA layer='data' (LogNormalize) CSV, containing:
  individual, label (HTS=1/LTS=0), LCORL, SFMBT2, ZFP36L2.

D  Descriptive pooled single-gene ROC (raw LogNormalize expression as scores).
E  Nine-classifier per-donor AUROC (outer leave-one-donor-out).
F  Pooled out-of-fold ROC, all nine classifiers.
G  Pooled out-of-fold precision-recall, all nine classifiers.
H  CatBoost three-gene ablation: paired per-donor delta AP.

Outer CV = leave-one-donor-out (LODO); inner = 4-fold GroupKFold by donor.
Tuning: mean held-out donor AP, then donor AUROC, with deterministic ties.
Weights: inverse donor-by-class counts for estimators supporting sample_weight.
KNN and LDA do not accept sample_weight and are fitted unweighted.

IMPORTANT: The three genes were selected using this same nine-donor cohort.
Results are post-selection INTERNAL performance assessments, NOT external
validation; they do not eliminate feature-selection / signature-label circularity.
Do not manually force the old AUC=0.732, AP=0.702 values into these figures.

Dependencies: numpy pandas matplotlib scikit-learn xgboost catboost.
Jupyter: %run Figure6_3genes_Nested_LODO_SVG_E060_100_FG_uniform.py
"""
from __future__ import annotations

from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # export without requiring an interactive desktop
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError(
        '缺少 xgboost 或 catboost，请在 Jupyter 中运行：%pip install xgboost catboost'
    ) from exc

# ======================== 只需修改此配置区 ========================
PROJECT_DIR = Path('/home/t440/machine/machine2')
DATA_FILE = PROJECT_DIR / 'AD_microglia_HTS_LTS_TWAS_genes_ML.csv'
OUTPUT_DIR = PROJECT_DIR / 'Figure6_3gene_nested_LODO'
GENES = ['LCORL', 'SFMBT2', 'ZFP36L2']
SEED = 42
INNER_SPLITS = 4
N_JOBS = 2  # 老电脑可设为 1；减少同时运行的 CPU 线程
ABLATION_MODEL = 'CatBoost'  # 预先固定，不能按外层测试成绩临时选择
# ===============================================================

MODEL_ORDER = ['KNN', 'LDA', 'LR', 'NB', 'RF', 'RPART', 'SVM', 'XGBoost', 'CatBoost']
COLOR_LIST = [
    '#4878A8', '#9C755F', '#4D908E', '#B07AA1', '#F28E2B',
    '#7F7F7F', '#59A14F', '#E15759', '#6C5B9A'
]
MODEL_COLORS = dict(zip(MODEL_ORDER, COLOR_LIST))
# Three-color palette, assigned to the three genes in GENES order.
STATE_COLORS = {'HTS': '#D88F91', 'MTS': '#BDBDBD', 'LTS': '#80BCC8'}
GENE_COLORS = dict(zip(GENES, [STATE_COLORS['HTS'], STATE_COLORS['MTS'], STATE_COLORS['LTS']]))

# 小型、可复现的参数网格。它们是本次新评估的网格，未声称与旧 Fig.6 代码相同。
PARAM_GRIDS = {
    'KNN': {'n_neighbors': [5, 11], 'weights': ['uniform', 'distance']},
    'LDA': {'solver': ['svd', 'lsqr']},
    'LR': {'C': [0.1, 1.0, 10.0]},
    'NB': {'var_smoothing': [1e-9, 1e-7]},
    'RF': {'n_estimators': [150], 'max_depth': [None, 6], 'min_samples_leaf': [2, 5]},
    'RPART': {'max_depth': [3, 6], 'min_samples_leaf': [5, 15]},
    'SVM': {'C': [0.1, 1.0, 10.0], 'gamma': ['scale']},
    'XGBoost': {'n_estimators': [100, 200], 'learning_rate': [0.03, 0.1],
                 'max_depth': [2, 3]},
    'CatBoost': {'iterations': [120, 240], 'depth': [3, 5],
                 'learning_rate': [0.05]},
}


def read_input(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f'输入文件不存在：{path}')
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    # R write.csv(row.names=TRUE) may add an Unnamed index column; ignore it.
    unwanted = [c for c in df.columns if c.startswith('Unnamed:')]
    df = df.drop(columns=unwanted)
    required = ['individual', 'label'] + GENES
    missing = [g for g in required if g not in df.columns]
    if missing:
        raise ValueError(f'缺少必要列：{missing}；实际列名示例：{list(df.columns[:15])}')
    if df['individual'].isna().any():
        raise ValueError('individual 存在缺失供体编号')
    y = pd.to_numeric(df['label'], errors='raise').to_numpy()
    if not np.isin(y, [0, 1]).all() or set(y) != {0, 1}:
        raise ValueError('label 只能包含 HTS=1 和 LTS=0，且两类都存在')
    y = y.astype(int)
    X = df[GENES].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError('三基因表达存在 NaN、Inf 或 -Inf')
    groups = df['individual'].astype(str).to_numpy()
    donors = np.sort(np.unique(groups))
    if len(donors) < INNER_SPLITS + 1:
        raise ValueError(f'至少需要 {INNER_SPLITS + 1} 名供体才可做外层 LODO + 内层分组 CV')
    for donor in donors:
        if np.unique(y[groups == donor]).size != 2:
            raise ValueError(f'供体 {donor} 缺少 HTS 或 LTS，无法计算其 ROC/AP')
    if len(donors) != 9:
        warnings.warn(f'检测到 {len(donors)} 名供体，不是正文描述的 9 名；请核对数据。')
    df = df.reset_index(drop=True)
    return df, X.reset_index(drop=True), y, groups, donors


def donor_class_weights(groups, y):
    counts = pd.DataFrame({'donor': groups, 'label': y}).groupby(['donor', 'label']).size()
    w = np.array([1.0 / counts.loc[(g, int(label))] for g, label in zip(groups, y)])
    return w / w.mean()


def make_estimator(name, params):
    p = dict(params)
    if name == 'KNN':
        return Pipeline([('scale', StandardScaler()),
                         ('model', KNeighborsClassifier(**p))])
    if name == 'LDA':
        return Pipeline([('scale', StandardScaler()),
                         ('model', LinearDiscriminantAnalysis(**p))])
    if name == 'LR':
        return Pipeline([('scale', StandardScaler()),
                         ('model', LogisticRegression(max_iter=3000, random_state=SEED, **p))])
    if name == 'NB':
        return GaussianNB(**p)
    if name == 'RF':
        return RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS, **p)
    if name == 'RPART':
        return DecisionTreeClassifier(random_state=SEED, **p)
    if name == 'SVM':
        return Pipeline([('scale', StandardScaler()),
                         ('model', SVC(kernel='rbf', probability=True,
                                       random_state=SEED, cache_size=250, **p))])
    if name == 'XGBoost':
        return XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                             tree_method='hist', random_state=SEED, n_jobs=N_JOBS,
                             subsample=0.8, colsample_bytree=0.8, **p)
    if name == 'CatBoost':
        return CatBoostClassifier(loss_function='Logloss', random_seed=SEED,
                                  thread_count=N_JOBS, verbose=False,
                                  allow_writing_files=False, **p)
    raise KeyError(f'未知模型：{name}')


def fit_estimator(name, estimator, X, y, groups):
    weights = donor_class_weights(groups, y)
    if name in ('KNN', 'LDA'):
        estimator.fit(X, y)  # sklearn KNN / LDA 不支持 fit(sample_weight=...)
    elif isinstance(estimator, Pipeline):
        estimator.fit(X, y, model__sample_weight=weights)
    else:
        estimator.fit(X, y, sample_weight=weights)
    return estimator


def positive_score(estimator, X):
    probs = np.asarray(estimator.predict_proba(X))
    if probs.ndim != 2 or probs.shape[1] != 2:
        raise ValueError('预期二分类模型返回 n_cells × 2 的概率矩阵')
    # sklearn and boosted model classes are ordered [0, 1] here.
    return probs[:, 1].astype(float)


def tune_on_development(name, X_dev, y_dev, g_dev, test_donor):
    """The held-out donor is NEVER used here (including preprocessing)."""
    fold_splitter = GroupKFold(n_splits=INNER_SPLITS)
    tuning_rows = []
    for param_index, params in enumerate(ParameterGrid(PARAM_GRIDS[name])):
        ap_by_donor, auc_by_donor = [], []
        for train_idx, val_idx in fold_splitter.split(X_dev, y_dev, g_dev):
            fitted = fit_estimator(
                name, make_estimator(name, params),
                X_dev.iloc[train_idx], y_dev[train_idx], g_dev[train_idx])
            val_prob = positive_score(fitted, X_dev.iloc[val_idx])
            for donor in np.unique(g_dev[val_idx]):
                mask = g_dev[val_idx] == donor
                actual = y_dev[val_idx][mask]
                score = val_prob[mask]
                if len(np.unique(actual)) != 2:
                    raise ValueError(f'内层供体 {donor} 只有单一类别')
                ap_by_donor.append(average_precision_score(actual, score))
                auc_by_donor.append(roc_auc_score(actual, score))
        tuning_rows.append({
            'model': name, 'outer_test_donor': str(test_donor),
            'param_index': param_index,
            'params_json': json.dumps(params, sort_keys=True, ensure_ascii=False),
            'inner_mean_donor_AP': float(np.mean(ap_by_donor)),
            'inner_mean_donor_AUROC': float(np.mean(auc_by_donor)),
            'n_validation_donors': len(ap_by_donor),
        })
    # Stable tie-breaking: highest donor AP, highest donor AUROC, then first grid entry.
    ranking = sorted(tuning_rows, key=lambda r: (
        -r['inner_mean_donor_AP'], -r['inner_mean_donor_AUROC'], r['param_index']))
    return json.loads(ranking[0]['params_json']), tuning_rows


def outer_run(name, X, y, groups, donors, feature_names):
    X = X.loc[:, feature_names].copy()
    predictions, performance, tuning = [], [], []
    for fold, donor in enumerate(donors, start=1):
        outer_test = groups == donor
        outer_dev = ~outer_test
        X_dev, X_test = X.loc[outer_dev].reset_index(drop=True), X.loc[outer_test]
        y_dev, y_test = y[outer_dev], y[outer_test]
        g_dev = groups[outer_dev]
        params, tuning_rows = tune_on_development(name, X_dev, y_dev, g_dev, donor)
        tuning.extend(tuning_rows)
        clf = fit_estimator(name, make_estimator(name, params), X_dev, y_dev, g_dev)
        score = positive_score(clf, X_test)
        auc = roc_auc_score(y_test, score)
        ap = average_precision_score(y_test, score)
        performance.append({
            'model': name, 'feature_set': '+'.join(feature_names),
            'fold': fold, 'test_donor': str(donor),
            'n_test_cells': len(y_test), 'n_HTS': int(y_test.sum()),
            'n_LTS': int((y_test == 0).sum()),
            'HTS_prevalence': float(y_test.mean()),
            'AUROC': float(auc), 'AP': float(ap),
            'best_params_json': json.dumps(params, sort_keys=True),
        })
        for i, label, prob in zip(np.flatnonzero(outer_test), y_test, score):
            predictions.append({
                'model': name, 'feature_set': '+'.join(feature_names),
                'fold': fold, 'row_index': int(i), 'individual': str(donor),
                'true_label': int(label), 'predicted_probability': float(prob),
            })
        print(f'  Fold {fold}/{len(donors)} donor={donor}: AUROC={auc:.3f}, AP={ap:.3f}',
              flush=True)
    return pd.DataFrame(performance), pd.DataFrame(predictions), pd.DataFrame(tuning)


def config_plot():
    plt.rcParams.update({
        'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.labelsize': 10,
        'axes.titlesize': 11, 'axes.linewidth': 0.9,
        'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
        'legend.fontsize': 8, 'pdf.fonttype': 42, 'ps.fonttype': 42,
        'savefig.facecolor': 'white', 'figure.facecolor': 'white',
        'svg.fonttype': 'none',
    })


def style_axes(ax, xlabel, ylabel, label):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(label, loc='left', weight='bold', pad=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(length=3)
    ax.grid(axis='y', color='#D9D9D9', lw=0.6, alpha=0.55)
    ax.set_axisbelow(True)


def save_figure(fig, name, tight=True):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # Keep identical SVG canvas dimensions for Figure 6F and 6G.
    kwargs = {'bbox_inches': 'tight'} if tight else {}
    fig.savefig(OUTPUT_DIR / f'{name}.svg', format='svg', facecolor='white', **kwargs)
    plt.close(fig)
    print(' Saved:', name + '.svg')


def figure_d(X, y, groups, donors):
    # Direct raw expression scores, not a cross-validated fitted gene classifier.
    donor_rows = []
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    for gene in GENES:
        score = X[gene].to_numpy()
        auc = roc_auc_score(y, score)
        fpr, tpr, _ = roc_curve(y, score)
        ax.plot(fpr, tpr, color=GENE_COLORS[gene], lw=2.1,
                label=f'{gene}  (AUC {auc:.3f})')
        for donor in donors:
            m = groups == donor
            donor_rows.append({'gene': gene, 'donor': donor,
                               'AUROC': roc_auc_score(y[m], score[m]),
                               'n_cells': int(m.sum())})
    ax.plot([0, 1], [0, 1], ls='--', color='#A0A0A0', lw=1)
    ax.set(xlim=(0, 1), ylim=(0, 1.01))
    style_axes(ax, 'False positive rate', 'True positive rate',
               'D  Single-gene discrimination')
    ax.legend(loc='lower right', frameon=False)
    fig.tight_layout()
    save_figure(fig, 'Figure_6D_three_single_gene_ROC')
    pd.DataFrame(donor_rows).to_csv(OUTPUT_DIR/'Figure_6D_gene_donor_AUROC.csv', index=False)


def figure_e(perf):
    names = MODEL_ORDER
    distributions = [perf.loc[perf.model == n, 'AUROC'].to_numpy() for n in names]
    fig, ax = plt.subplots(figsize=(9.2, 4.5))
    boxes = ax.boxplot(distributions, patch_artist=True, widths=0.55,
                       showfliers=False, whis=(0, 100),
                       medianprops={'color': '#202020', 'linewidth': 1.4})
    for i, (box, values) in enumerate(zip(boxes['boxes'], distributions)):
        box.set(facecolor=MODEL_COLORS[names[i]], alpha=0.18,
                edgecolor=MODEL_COLORS[names[i]], linewidth=1.3)
        rng = np.random.default_rng(SEED + i)
        xj = (i + 1) + rng.uniform(-0.10, 0.10, size=len(values))
        ax.scatter(xj, values, s=23, alpha=0.9, color=MODEL_COLORS[names[i]],
                   linewidth=0, zorder=3)
    ax.set_xticks(range(1, len(names) + 1), names, rotation=30, ha='right')
    # Requested 0.60–1.00 scale (held-out donor AUROC).
    if any((values < 0.60).any() for values in distributions):
        warnings.warn('有供体 AUROC 小于 0.60；固定坐标轴将使对应散点无法显示。')
    ax.set_ylim(0.60, 1.00)
    ax.set_yticks([0.60, 0.70, 0.80, 0.90, 1.00])
    style_axes(ax, 'Classifier', 'Held-out donor AUROC',
               'E  Three-gene model comparison')
    fig.tight_layout()
    save_figure(fig, 'Figure_6E_nine_model_donor_AUROC')


def figure_f_and_g(pred):
    """Both SVGs have identical canvas sizes and equal physical axis lengths."""
    curves = {}
    for name in MODEL_ORDER:
        part = pred[pred.model == name]
        y = part.true_label.to_numpy()
        s = part.predicted_probability.to_numpy()
        fpr, tpr, _ = roc_curve(y, s)
        precision, recall, _ = precision_recall_curve(y, s)
        curves[name] = {'fpr': fpr, 'tpr': tpr, 'precision': precision,
                        'recall': recall, 'auc': roc_auc_score(y, s),
                        'ap': average_precision_score(y, s)}

    # Fixed matched layout for F and G. Both plot squares are 3.92 x 3.92 inches.
    # Legend outside the square does not affect its physical side lengths.
    fg_figsize = (7.4, 5.6)
    fg_axes_rect = [0.13, 0.16, 0.53, 0.70]

    fig, ax = plt.subplots(figsize=fg_figsize)
    ax.set_position(fg_axes_rect)
    for name in MODEL_ORDER:
        c = curves[name]
        ax.plot(c['fpr'], c['tpr'], color=MODEL_COLORS[name], lw=1.8,
                label=f'{name}  {c["auc"]:.3f}')
    ax.plot([0, 1], [0, 1], ls='--', color='#999999', lw=0.9)
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_box_aspect(1)
    style_axes(ax, '1 - Specificity', 'Sensitivity',
               'F  Pooled out-of-fold ROC')
    ax.legend(loc='center left', bbox_to_anchor=(1.03, 0.5),
              frameon=False, title='AUROC', fontsize=8,
              title_fontsize=9, labelspacing=0.42)
    save_figure(fig, 'Figure_6F_nine_model_pooled_ROC', tight=False)

    fig, ax = plt.subplots(figsize=fg_figsize)
    ax.set_position(fg_axes_rect)
    for name in MODEL_ORDER:
        c = curves[name]
        ax.plot(c['recall'], c['precision'], color=MODEL_COLORS[name], lw=1.8,
                label=f'{name}  {c["ap"]:.3f}')
    base_pred = pred[pred.model == MODEL_ORDER[0]]
    prevalence = base_pred.true_label.mean()
    ax.axhline(prevalence, ls='--', color='#999999', lw=1,
               label=f'HTS prevalence  {prevalence:.3f}')
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_box_aspect(1)
    style_axes(ax, 'Recall', 'Precision', 'G  Pooled out-of-fold PR')
    ax.legend(loc='center left', bbox_to_anchor=(1.03, 0.5),
              frameon=False, title='AP', fontsize=8,
              title_fontsize=9, labelspacing=0.36)
    save_figure(fig, 'Figure_6G_nine_model_pooled_PR', tight=False)
    return curves


def figure_h(ablation_delta):
    display = [('LCORL', 'Without LCORL'),
               ('SFMBT2', 'Without SFMBT2'),
               ('ZFP36L2', 'Without ZFP36L2')]
    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    for i, (gene, label) in enumerate(display, start=1):
        sub = ablation_delta[ablation_delta.removed_gene == gene].copy()
        values = sub.delta_AP.to_numpy()
        # jitter only in x; true delta_AP values remain unchanged
        rng = np.random.default_rng(SEED + i)
        xj = i + rng.uniform(-0.09, 0.09, size=len(values))
        ax.scatter(xj, values, color=GENE_COLORS[gene], s=30, alpha=0.86,
                   edgecolor='white', linewidth=0.35, zorder=3)
        med = np.median(values)
        q1, q3 = np.quantile(values, [0.25, 0.75])
        ax.vlines(i, q1, q3, color='#1C1C1C', lw=4, zorder=4)
        ax.hlines(med, i - 0.17, i + 0.17, color='#1C1C1C', lw=2.0, zorder=5)
    ax.axhline(0, color='#666666', lw=1.2, ls='--')
    ax.set_xticks([1, 2, 3], [x[1] for x in display])
    style_axes(ax, 'Ablated feature',
               r'$\Delta$AP = AP(3 genes) - AP(2 genes)',
               f'H  {ABLATION_MODEL} three-gene ablation')
    ax.text(0.02, 0.98, 'Positive values favor the complete panel',
            transform=ax.transAxes, ha='left', va='top', fontsize=8,
            color='#444444')
    fig.tight_layout()
    save_figure(fig, 'Figure_6H_three_gene_ablation_delta_AP')


def run_all():
    config_plot()
    df, X, y, groups, donors = read_input(DATA_FILE)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if ABLATION_MODEL not in MODEL_ORDER:
        raise ValueError('ABLATION_MODEL 必须是九种模型之一')
    print('Input:', DATA_FILE, '\nOutput:', OUTPUT_DIR)
    print('Cells:', len(y), 'Donors:', len(donors), 'HTS prevalence:', round(y.mean(), 3))
    print('Genes:', GENES)
    print('WARNING: post-selection internal performance; no external validation.')
    print('KNN / LDA fit without donor-class weights (API limitation).')
    print('Starting Figure 6D single-gene ROC...', flush=True)
    figure_d(X, y, groups, donors)

    perf_frames, pred_frames, tuning_frames = [], [], []
    for model in MODEL_ORDER:
        print(f'\n[{model}] Nested LODO, 3-gene panel...', flush=True)
        performance, preds, tunings = outer_run(model, X, y, groups, donors, GENES)
        perf_frames.append(performance)
        pred_frames.append(preds)
        tuning_frames.append(tunings)
        # Checkpoint after EACH classifier to preserve partial results if interrupted.
        pd.concat(perf_frames, ignore_index=True).to_csv(
            OUTPUT_DIR/'Figure_6_full_model_donor_metrics_IN_PROGRESS.csv', index=False)
        pd.concat(pred_frames, ignore_index=True).to_csv(
            OUTPUT_DIR/'Figure_6_full_model_OOF_predictions_IN_PROGRESS.csv', index=False)

    perf = pd.concat(perf_frames, ignore_index=True)
    pred = pd.concat(pred_frames, ignore_index=True)
    tuning = pd.concat(tuning_frames, ignore_index=True)
    perf.to_csv(OUTPUT_DIR/'Figure_6_full_model_donor_metrics.csv', index=False)
    pred.to_csv(OUTPUT_DIR/'Figure_6_full_model_OOF_predictions.csv', index=False)
    tuning.to_csv(OUTPUT_DIR/'Figure_6_full_model_inner_tuning.csv', index=False)
    summary = []
    for model in MODEL_ORDER:
        d = perf[perf.model == model]
        p = pred[pred.model == model]
        summary.append({
            'model': model, 'n_donors': len(d),
            'median_donor_AUROC': d.AUROC.median(),
            'AUROC_Q1': d.AUROC.quantile(.25), 'AUROC_Q3': d.AUROC.quantile(.75),
            'median_donor_AP': d.AP.median(),
            'AP_Q1': d.AP.quantile(.25), 'AP_Q3': d.AP.quantile(.75),
            'pooled_OOF_AUROC': roc_auc_score(p.true_label, p.predicted_probability),
            'pooled_OOF_AP': average_precision_score(p.true_label, p.predicted_probability),
        })
    summary = pd.DataFrame(summary)
    summary.to_csv(OUTPUT_DIR/'Figure_6_nine_model_summary.csv', index=False)
    # Manuscript-ready supplementary tables (one CSV per table).
    summary.to_csv(OUTPUT_DIR/'Supplementary_Table_S12.csv', index=False)
    selected = perf[['model', 'test_donor', 'best_params_json']].copy()
    tuning_s13 = tuning.merge(selected, left_on=['model', 'outer_test_donor'],
                              right_on=['model', 'test_donor'], how='left')
    tuning_s13['selected_in_outer_fold'] = (
        tuning_s13['params_json'] == tuning_s13['best_params_json'])
    tuning_s13.drop(columns=['test_donor', 'best_params_json']).to_csv(
        OUTPUT_DIR/'Supplementary_Table_S13.csv', index=False)
    figure_e(perf)
    figure_f_and_g(pred)

    # Prespecified CatBoost ablation. Reuse full-panel CatBoost outer CV values.
    reference = perf[perf.model == ABLATION_MODEL].set_index('test_donor')
    ablation_metrics, ablation_predictions, ablation_tuning = [], [], []
    delta_rows = []
    for removed in GENES:
        retained = [g for g in GENES if g != removed]
        print(f'\n[Ablation] {ABLATION_MODEL}, without {removed}...', flush=True)
        met, pred_abl, tuning_abl = outer_run(ABLATION_MODEL, X, y, groups, donors, retained)
        met['removed_gene'] = removed
        pred_abl['removed_gene'] = removed
        tuning_abl['removed_gene'] = removed
        ablation_metrics.append(met)
        ablation_predictions.append(pred_abl)
        ablation_tuning.append(tuning_abl)
        for row in met.itertuples(index=False):
            full = reference.loc[str(row.test_donor)]
            delta_rows.append({
                'model': ABLATION_MODEL, 'donor': row.test_donor,
                'removed_gene': removed, 'n_test_cells': row.n_test_cells,
                'AP_full_3genes': float(full['AP']), 'AP_without_gene': row.AP,
                'delta_AP': float(full['AP'] - row.AP),
                'AUROC_full_3genes': float(full['AUROC']),
                'AUROC_without_gene': row.AUROC,
                'delta_AUROC': float(full['AUROC'] - row.AUROC),
            })
    ablation = pd.DataFrame(delta_rows)
    ablation.to_csv(OUTPUT_DIR/'Figure_6H_ablation_per_donor_delta_AP.csv', index=False)
    ablation.to_csv(OUTPUT_DIR/'Supplementary_Table_S14.csv', index=False)
    pd.concat(ablation_metrics, ignore_index=True).to_csv(
        OUTPUT_DIR/'Figure_6H_ablation_donor_metrics.csv', index=False)
    pd.concat(ablation_predictions, ignore_index=True).to_csv(
        OUTPUT_DIR/'Figure_6H_ablation_OOF_predictions.csv', index=False)
    pd.concat(ablation_tuning, ignore_index=True).to_csv(
        OUTPUT_DIR/'Figure_6H_ablation_inner_tuning.csv', index=False)
    figure_h(ablation)

    report = {
        'input_file': str(DATA_FILE), 'output_directory': str(OUTPUT_DIR),
        'data': 'Seurat RNA LogNormalize; features are not re-normalized globally',
        'class': {'HTS': 1, 'LTS': 0}, 'genes': GENES,
        'n_cells': len(y), 'donors': [str(x) for x in donors],
        'outer_cv': 'leave-one-donor-out', 'inner_cv': f'{INNER_SPLITS}-fold GroupKFold',
        'optimization': 'mean donor AP then mean donor AUROC',
        'ablation_model_preselected': ABLATION_MODEL,
        'training_weights': 'inverse donor x class, normalized to mean 1; KNN/LDA unweighted',
        'figure_6D': 'raw single-gene pooled ROC; descriptive, no fitted model',
        'figure_6E': 'one AUROC dot per held-out donor per model',
        'figure_6F_G': 'pooled out-of-fold predictions, not donor medians',
        'figure_6H': 'paired AP difference on same held-out donors',
        'caution': 'same donor cohort used for gene selection and model evaluation; '
                   'post-selection internal benchmark only; potential circularity with HTS/LTS signature',
        'parameter_grids': PARAM_GRIDS,
    }
    with open(OUTPUT_DIR/'Figure_6_methods_and_config.json', 'w', encoding='utf-8') as fh:
        json.dump(report, fh, ensure_ascii=False, indent=2)

    print('\n' + '='*65 + '\nDONE: 5 figures (D/E/F/G/H), SVG only, and CSV results including S12-S14 (no S11).')
    print('Saved under:', OUTPUT_DIR)
    print('\nNine models, actual observed metrics (do not reuse previous values):')
    print(summary[['model', 'median_donor_AUROC', 'median_donor_AP',
                   'pooled_OOF_AUROC', 'pooled_OOF_AP']].round(3).to_string(index=False))
    print('\nAblation median delta AP:')
    print(ablation.groupby('removed_gene')['delta_AP'].median().round(4).to_string())
    return summary, ablation


if __name__ == '__main__':
    try:
        run_all()
    except (FileNotFoundError, ValueError, KeyError) as exc:
        print('ERROR:', exc, file=sys.stderr)
        print('请检查 DATA_FILE、输入列名和已安装的依赖包。', file=sys.stderr)
        # Do not raise SystemExit in Jupyter; avoid its long exception formatting.


Input: /home/t440/machine/machine2/AD_microglia_HTS_LTS_TWAS_genes_ML.csv 
Output: /home/t440/machine/machine2/Figure6_3gene_nested_LODO
Cells: 4139 Donors: 9 HTS prevalence: 0.54
Genes: ['LCORL', 'SFMBT2', 'ZFP36L2']
KNN / LDA fit without donor-class weights (API limitation).
Starting Figure 6D single-gene ROC...
 Saved: Figure_6D_three_single_gene_ROC.svg

[KNN] Nested LODO, 3-gene panel...
  Fold 1/9 donor=A019/12: AUROC=0.625, AP=0.672
  Fold 2/9 donor=A096/14: AUROC=0.750, AP=0.777
  Fold 3/9 donor=A163/17: AUROC=0.732, AP=0.620
  Fold 4/9 donor=A169/10: AUROC=0.658, AP=0.790
  Fold 5/9 donor=A170/13: AUROC=0.718, AP=0.687
  Fold 6/9 donor=A272/15: AUROC=0.702, AP=0.686
  Fold 7/9 donor=A277/12: AUROC=0.679, AP=0.727
  Fold 8/9 donor=A282/14: AUROC=0.650, AP=0.436
  Fold 9/9 donor=A402/14: AUROC=0.682, AP=0.638

[LDA] Nested LODO, 3-gene panel...
  Fold 1/9 donor=A019/12: AUROC=0.667, AP=0.708
  Fold 2/9 donor=A096/14: AUROC=0.776, AP=0.829
  Fold 3/9 donor=A163/17: AUROC=0.761, A